# transformers / GPT-2 smoke test

Mirrors what UCSC's computational linguistics course does: load a GPT-2
model, inspect its layers and parameters, and run a short generation.

A **tiny randomly-initialised** GPT-2 is built from a config rather than
downloading the 124M-parameter checkpoint — this test must not depend on
network access to huggingface.co, and the point is to prove the
torch + transformers path works, not to reproduce the real weights.

In [ ]:
import torch
import transformers
from transformers import GPT2Config, GPT2LMHeadModel

print("torch", torch.__version__)
print("transformers", transformers.__version__)

In [ ]:
# Small on purpose: real GPT-2 is n_layer=12, n_embd=768.
config = GPT2Config(n_layer=2, n_head=2, n_embd=64, vocab_size=256, n_positions=64)
model = GPT2LMHeadModel(config)
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(f"{n_params:,} parameters across {len(list(model.parameters()))} tensors")
assert n_params > 0

In [ ]:
# Inspecting layers is the other half of what the course does.
blocks = model.transformer.h
print(f"{len(blocks)} transformer blocks")
for name, _ in list(blocks[0].named_parameters())[:4]:
    print("  ", name)
assert len(blocks) == 2

In [ ]:
# A short generation, the way a prompt demo would run.
prompt = torch.randint(0, config.vocab_size, (1, 4))
with torch.no_grad():
    out = model.generate(prompt, max_new_tokens=8, do_sample=False,
                         pad_token_id=config.vocab_size - 1)

print("input", tuple(prompt.shape), "-> output", tuple(out.shape))
assert out.shape[1] == prompt.shape[1] + 8